In [ ]:
# fix imports
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

# change dir to ..
os.chdir(module_path)

In [ ]:
from src.adv_model import AdvModel
import torch
from scripts.utils.load_model import print_supported_models, load_model

torch.set_float32_matmul_precision("high")

model, tokenizer = load_model("meta-llama/Llama-2-7b-chat-hf", torch_dtype="bfloat16")

In [ ]:
from src.utils import env
from src.initialize import Initializer

# env.set_seed(41)

adv_model = AdvModel(
    model=model,
    tokenizer=tokenizer,
    num_tokens=20,
)

Initializer.normal(adv_model)

In [ ]:
from src.sample_attacks.soft_prompt import SoftPrompt
from src.sample_attacks import PEZ
from torch import optim
from src.config import GenConfig

# sample_attack = SoftPrompt(
#     adv_model,
#     optim_factory=lambda params: optim.AdamW(params, lr=1e-3),
#     steps=200,
#     mixed_precision=False,
#     kv_caching=True,
#     early_stopping=True,
# )

sample_attack = PEZ(
    adv_model,
    steps=10,
    optim_factory=lambda params: optim.Adam(params, lr=0.01),
    mixed_precision=False,
    early_stopping=True,
)

In [ ]:
from src.sample_attacks.harm_bench.pez import PEZ as HarmBenchPEZ

hb_pez = HarmBenchPEZ(
    adv_model,
    num_optim_tokens=20,
    num_steps=10,
    lr=0.01,
    verbose=True,
)

In [ ]:
from scripts.utils.load_dataset import load_datasets

ds_train, ds_val, ds_test = load_datasets(["harmbench"], val_size=0)

inputs = ds_train["prompt"].tolist()
targets = ds_train["target"].tolist()

convs = [[{"role": "user", "content": inp}] for inp in inputs]

In [ ]:
# work in batches of batch_size
from tqdm.auto import tqdm

config = GenConfig(
    max_new_tokens=256,
    do_sample=False,
)

batch_size = 5

# create 2 columns in ds_train: preds_new, preds_hb

ds_train["preds_new"] = ""
ds_train["preds_hb"] = ""

for i in tqdm(range(0, len(convs), batch_size)):
    convs_batch = convs[i : i + batch_size]
    targets_batch = targets[i : i + batch_size]

    attack_result = sample_attack.fit(convs_batch, targets_batch)
    # hb_attack_result = hb_pez.fit(convs_batch, targets_batch)
    
    preds = adv_model.chat(attack_result.conversations, config=config, adv_embeds=attack_result.adv_embeds)
    # preds_hb = adv_model.chat(hb_attack_result.conversations, config=config, adv_embeds=hb_attack_result.adv_embeds)
    
    ds_train.loc[ds_train.index[i : i + batch_size], "preds_new"] = preds
    # ds_train.loc[ds_train.index[i : i + batch_size], "preds_hb"] = preds_hb

In [ ]:
from src.eval import HarmBenchJudge
from gserve import ServeConfig, LLMConfig


judge = HarmBenchJudge(ServeConfig(gpu_ids=[1]))

In [ ]:
from src.data import TableLoader

dl_train = TableLoader(ds_train, batch_size=50, shuffle=False)

dl_train.df["response"] = dl_train.df["preds_new"]
res1 = judge.evaluate(dl_train)
print(res1)

In [ ]:
# dl_train.df["response"] = dl_train.df["preds_hb"]
# res2 = judge.evaluate(dl_train)
# print(res2)

In [ ]:
i = 10

targets_batch = targets[i : i + batch_size]
convs_batch = convs[i : i + batch_size]

sample_attack = PEZ(
    adv_model,
    steps=100,
    optim_factory=lambda params: optim.Adam(params, lr=0.01),
    mixed_precision=False,
    early_stopping=True,
)

attack_result = sample_attack.fit(convs_batch, targets_batch)
# hb_attack_result = hb_pez.fit(convs_batch, targets_batch)

preds = adv_model.chat(attack_result.conversations, config=config, adv_embeds=attack_result.adv_embeds)

In [ ]:
# print convo, target, pred

for c, t, p in zip(attack_result.conversations, targets_batch, preds):
    print("CONV:", c)
    print("TARGET:", t)
    print("PRED:", p)
    print("===")